# 07 · Reading a plate by redrawing it

The YOLO26s reader is kept as it is. After it, the 5 best valid readings of a plate are drawn as clean plates,
fitted to the photo (corners, brightness, blur), and the reading whose drawing fits best is chosen.

**Inputs:** the `bangla-plate-reading` dataset and the YOLO26s reader `best.pt`.

In [ ]:
%pip install -q ultralytics==8.4.46

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

DATA = next(p for p in INPUT.rglob("bangla-plate-reading") if p.is_dir())
WEIGHTS = list(INPUT.rglob("best.pt"))
print(DATA)
print(WEIGHTS)

In [ ]:
classes = json.loads((DATA / "classes.json").read_text(encoding="utf-8"))
bangla = [c["bangla"] for c in classes]
kind = [c["kind"] for c in classes]

test_images = sorted((DATA / "images" / "test").iterdir())
print(len(classes), "classes")
print(len(test_images), "test plates")

## Plate text

Same as notebooks 05 and 06: district, word and letter from left to right, then the six digits as `xx-xxxx`.
The true text of a test plate comes from its label file.

In [ ]:
def plate_text(chars):
    # chars: list of (class id, x centre)
    chars = sorted(chars, key=lambda c: c[1])
    head = " ".join(bangla[c] for c, x in chars if kind[c] != "digit")
    digits = "".join(bangla[c] for c, x in chars if kind[c] == "digit")
    if len(digits) == 6:
        digits = digits[:2] + "-" + digits[2:]
    return " ".join(part for part in [head, digits] if part)


def edit_distance(a, b):
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            prev, row[j] = row[j], min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
    return row[-1]


def read_labels(image_path, split="test"):
    # YOLO label file -> list of (class id, x centre, y centre, width, height)
    label = DATA / "labels" / split / (image_path.stem + ".txt")
    boxes = []
    for line in label.read_text().splitlines():
        p = line.split()
        if p:
            boxes.append((int(p[0]), *map(float, p[1:])))
    return boxes


def read_truth(image_path, split="test"):
    return plate_text([(b[0], b[1]) for b in read_labels(image_path, split)])


for path in test_images[:3]:
    print(path.name[:50], "->", read_truth(path))
print(edit_distance("ঢাকা মেট্রো গ ২৪-২৯৫৫", "ঢাকা মেট্রো ২৪-২৯৫৫"))

## Baseline: current reader on the test set

The reader at 0.25 confidence, characters sorted left to right, as in the deployed AI service.
This must match notebook 05: 77.4% on all 939 plates (75.7% bd-lpr, 77.9% blpd).

In [ ]:
reader = YOLO(WEIGHTS[0])
device = 0 if torch.cuda.is_available() else "cpu"

rows = []
for start in range(0, len(test_images), 64):
    batch = test_images[start:start + 64]
    results = reader.predict(batch, imgsz=320, conf=0.25, device=device, verbose=False)
    for path, r in zip(batch, results):
        chars = list(zip(r.boxes.cls.int().tolist(), r.boxes.xywhn[:, 0].tolist()))
        truth = read_truth(path)
        pred = plate_text(chars)
        rows.append({"image": path.name, "source": path.name.split("__")[0],
                     "truth": truth, "predicted": pred, "correct": truth == pred,
                     "errors": edit_distance(truth, pred), "length": len(truth)})

baseline = pd.DataFrame(rows)
baseline.head()

In [ ]:
def summary(df):
    return pd.Series({"plates": len(df),
                      "plate acc %": round(100 * df["correct"].mean(), 1),
                      "CER %": round(100 * df["errors"].sum() / df["length"].sum(), 2)})


table = baseline.groupby("source").apply(summary)
table.loc["all"] = summary(baseline)
table

## Plate rules: the 5 best valid readings

A Bangladeshi plate has one district, at most one word (Metro / Chatto / Raj), one series letter and six digits.

The reader is run again at 0.05 confidence so every character position keeps its weaker guesses too.
Guesses that overlap are one position. Each position either takes one of its (up to 3) guesses, scored `log(conf)`,
or is skipped as noise, scored `log(1 - best conf)`. A beam search keeps only readings that follow the plate pattern,
and the 5 best go to the redrawing step.

In [ ]:
def reader_guesses(images, conf=0.05):
    out = {}
    for start in range(0, len(images), 64):
        batch = images[start:start + 64]
        results = reader.predict(batch, imgsz=320, conf=conf, device=device, max_det=100, verbose=False)
        for path, r in zip(batch, results):
            out[path.name] = [
                {"cls": c, "conf": p, "x": xywh[0], "box": xyxy}
                for c, p, xywh, xyxy in zip(r.boxes.cls.int().tolist(), r.boxes.conf.tolist(),
                                            r.boxes.xywhn.tolist(), r.boxes.xyxyn.tolist())
            ]
    return out


test_guesses = reader_guesses(test_images)
print(sum(len(g) for g in test_guesses.values()), "guesses on", len(test_guesses), "plates")

In [ ]:
import math

LIMIT = {"digit": 6, "district": 1, "letter": 1, "word": 1}
KINDS = list(LIMIT)


def iou(a, b):
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union else 0.0


def positions(guesses):
    # overlapping guesses are one character position, best guess first
    groups = []
    for g in sorted(guesses, key=lambda g: -g["conf"]):
        for group in groups:
            if iou(group[0]["box"], g["box"]) > 0.5:
                group.append(g)
                break
        else:
            groups.append([g])
    return sorted(groups, key=lambda group: group[0]["x"])


def top_readings(guesses, k=5, beam=256):
    groups = positions(guesses)
    # a state is (score, counts per kind, chosen guesses)
    states = [(0.0, (0, 0, 0, 0), ())]
    for i, group in enumerate(groups):
        left = len(groups) - i - 1
        options = group[:3]
        new_states = []
        for score, counts, chosen in states:
            # skip this position
            new_states.append((score + math.log(max(1e-4, 1 - options[0]["conf"])), counts, chosen))
            # or take one of its guesses
            for g in options:
                j = KINDS.index(kind[g["cls"]])
                c = list(counts)
                c[j] += 1
                if c[j] <= LIMIT[KINDS[j]]:
                    new_states.append((score + math.log(max(1e-4, g["conf"])), tuple(c), chosen + (g,)))
        # drop states that can no longer reach six digits, a district and a letter
        new_states = [s for s in new_states if (6 - s[1][0]) + (1 - s[1][1]) + (1 - s[1][2]) <= left]
        states = sorted(new_states, key=lambda s: -s[0])[:beam]

    readings = {}
    for score, counts, chosen in states:
        if counts[:3] == (6, 1, 1):
            text = plate_text([(g["cls"], g["x"]) for g in chosen])
            if text not in readings or score > readings[text]["logp"]:
                readings[text] = {"text": text, "logp": score, "chars": [(g["cls"], g["box"]) for g in chosen]}
    return sorted(readings.values(), key=lambda r: -r["logp"])[:k]


example = test_images[0].name
for r in top_readings(test_guesses[example]):
    print(round(r["logp"], 2), r["text"])
print("truth:", read_truth(test_images[0]))

In [ ]:
rules = []
for path in test_images:
    readings = top_readings(test_guesses[path.name])
    truth = read_truth(path)
    pred = readings[0]["text"] if readings else ""
    rules.append({"image": path.name, "source": path.name.split("__")[0], "truth": truth,
                  "predicted": pred, "correct": truth == pred,
                  "errors": edit_distance(truth, pred), "length": len(truth),
                  "in_top5": any(r["text"] == truth for r in readings), "n_readings": len(readings)})
rules = pd.DataFrame(rules)

compare = pd.DataFrame({
    "baseline acc %": baseline.groupby("source")["correct"].mean() * 100,
    "plate rules acc %": rules.groupby("source")["correct"].mean() * 100,
    "truth in top 5 %": rules.groupby("source")["in_top5"].mean() * 100,
})
compare.loc["all"] = [baseline["correct"].mean() * 100, rules["correct"].mean() * 100, rules["in_top5"].mean() * 100]
print("plates with no valid reading:", (rules["n_readings"] == 0).sum())
compare.round(1)